# Model Evaluation & Analysis

This notebook provides comprehensive evaluation of the trained ML surrogate model including:
- Prediction accuracy metrics
- Residual analysis
- Feature importance
- Predictions vs Actual scatter plot

In [1]:
import sys, os
ROOT = os.path.abspath("..")   # go up one directory from notebooks/
if ROOT not in sys.path:
    sys.path.append(ROOT)

print(ROOT)  

c:\Users\Public\workspace\sh-wave-ml-surrogate


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

print("Libraries imported successfully!")

Libraries imported successfully!


## 1. Load Model and Data

In [3]:
import os
import pandas as pd
from joblib import load

ROOT = os.path.abspath("..")

# Load model
model = load(os.path.join(ROOT, 'data', 'models', 'gbr_model.pkl'))

# Load YOUR MATLAB data
data = pd.read_excel(os.path.join(ROOT, 'data', 'raw', 'dispersion_vs_L.xlsx'))

# Inputs
X_data = data[["kL", "L"]].values

# Output (IMPORTANT)
y_data = data["c_beta"].values

print(f"Model loaded: {type(model).__name__}")
print(f"Data shape: {X_data.shape}")

ImportError: `Import openpyxl` failed.  Use pip or conda to install the openpyxl package.

## 2. Regenerate Test Data with Known Targets

In [ ]:
from joblib import load

# Load scaler (VERY IMPORTANT)
scaler = load(os.path.join(ROOT, "data", "models", "scaler.pkl"))

# Scale your Excel inputs
X_scaled = scaler.transform(X_data)

# Use SAME data for testing
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_data, test_size=0.2, random_state=42
)

print("Test set size:", len(X_test))
print(f"y_test range: [{y_test.min():.4f}, {y_test.max():.4f}]")

## 3. Model Predictions

In [ ]:
# Generate predictions
y_pred = model.predict(X_test)

print(f"Predictions range: [{y_pred.min():.4f}, {y_pred.max():.4f}]")
print(f"First 10 predictions: {y_pred[:10]}")
print(f"First 10 actual values: {y_test[:10]}")

## 4. Evaluation Metrics

In [ ]:
# Calculate metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print("="*60)
print("MODEL EVALUATION METRICS")
print("="*60)
print(f"Mean Absolute Error (MAE):     {mae:.6f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.6f}")
print(f"R² Score:                      {r2:.6f}")
print(f"Mean Absolute % Error (MAPE):  {mape:.2f}%")
print("="*60)

# Interpretation
if r2 > 0.9:
    print("EXCELLENT: R² > 0.9 indicates very good model performance")
elif r2 > 0.7:
    print("GOOD: R² > 0.7 indicates good model performance")
else:
    print("MODERATE: R² < 0.7, model may need improvement")

## 5. Predictions vs Actual (Scatter Plot)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import load
from sklearn.metrics import r2_score

# ==============================
# 1. ROOT path
# ==============================
ROOT = os.path.abspath("..")

# ==============================
# 2. Load model + scaler
# ==============================
model = load(os.path.join(ROOT, "data", "models", "gbr_model.pkl"))
scaler = load(os.path.join(ROOT, "data", "models", "scaler.pkl"))

# ==============================
# 3. Load YOUR Excel data
# ==============================
data = pd.read_excel(os.path.join(ROOT, "data", "raw", "dispersion_vs_L.xlsx"))

X_data = data[["kL", "L"]].values
y_data = data["c_beta"].values

# ==============================
# 4. Scale
# ==============================
X_scaled = scaler.transform(X_data)

# ==============================
# 5. Predict
# ==============================
y_pred = model.predict(X_scaled)

# ==============================
# 6. Evaluate
# ==============================
r2 = r2_score(y_data, y_pred)

print(f"Model: {type(model).__name__}")
print(f"R² Score: {r2:.4f}")

# ==============================
# 7. Plot directory
# ==============================
plot_dir = os.path.join(ROOT, "results", "plots")
os.makedirs(plot_dir, exist_ok=True)

# ==============================
# 8. Plot (FINAL FIGURE)
# ==============================
plt.figure(figsize=(8, 6))

plt.scatter(y_data, y_pred, alpha=0.6, s=25)

# Perfect line
min_val = min(y_data.min(), y_pred.min())
max_val = max(y_data.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=1.5)

plt.xlabel(r'Analytical $c/\beta_l$', fontsize=12)
plt.ylabel(r'ML Predicted $c/\beta_l$', fontsize=12)
plt.title(f'Prediction Accuracy ($R^2$ = {r2:.4f})', fontsize=13)

plt.grid(True)
plt.tight_layout()

# ==============================
# 9. Save
# ==============================
plot_path = os.path.join(plot_dir, "ml_vs_analytical.png")
plt.savefig(plot_path, dpi=300)
plt.show()

print(f"Plot saved at: {plot_path}")

## 6. Residual Analysis

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split

ROOT = os.path.abspath("..")

PLOT_DIR = os.path.join(ROOT, "results", "plots")
os.makedirs(PLOT_DIR, exist_ok=True)

plot_path = os.path.join(PLOT_DIR, "residual_analysis.png")

# Recreate train-test split to align indices between raw and scaled data
X_raw = X_data  # unscaled inputs
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(X_raw, y_data, test_size=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_data, test_size=0.2, random_state=42)

# Compute predictions on the test set
y_pred_test = model.predict(X_test)
residuals = y_test - y_pred_test
errors = np.abs(residuals)

# Plots
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# (1) Residuals vs Predicted
axes[0, 0].scatter(y_pred_test, residuals, alpha=0.6, s=25)
axes[0, 0].axhline(y=0, linestyle='--', lw=1.5)
axes[0, 0].set_xlabel('Predicted $c/\\beta_l$'.replace('\\\\','\\'))
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Residuals vs Predicted')
axes[0, 0].grid(True)

# (2) Histogram
axes[0, 1].hist(residuals, bins=30)
axes[0, 1].set_xlabel('Residuals')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title(f'Mean = {residuals.mean():.6e}')
axes[0, 1].grid(True)

# (3) Q-Q Plot
stats.probplot(residuals, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot')

# (4) Error vs kL (use test set raw kL)
kL_vals = X_test_raw[:, 0]   # kL for test set
axes[1, 1].scatter(kL_vals, errors, alpha=0.6, s=25)
axes[1, 1].set_xlabel('$kL$')
axes[1, 1].set_ylabel('Absolute Error')
axes[1, 1].set_title('Error vs Wave Number')
axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig(plot_path, dpi=300)
plt.show()

print(f"Residual analysis plot saved at: {plot_path}")
print(f"Mean residual: {residuals.mean():.6e}")
print(f"Std of residuals: {residuals.std():.6e}")

## 7. Feature Importance

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

ROOT = os.path.abspath("..")

plot_dir = os.path.join(ROOT, "results", "plots")
os.makedirs(plot_dir, exist_ok=True)

plot_path = os.path.join(plot_dir, "feature_importance.png")

# ==============================
# Feature importance
# ==============================
if hasattr(model, 'feature_importances_'):
    
    # ✅ CORRECT features
    feature_names = [
        '$kL$ (Wave Number)',
        '$L$ (Layer Thickness)'
    ]
    
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1]

    plt.figure(figsize=(6, 5))
    
    plt.bar(range(len(importances)), importances[indices])
    
    plt.xticks(
        range(len(importances)),
        [feature_names[i] for i in indices]
    )
    
    plt.ylabel('Importance')
    plt.title('Feature Importance')
    plt.tight_layout()

    plt.savefig(plot_path, dpi=300)
    plt.show()

    print("Feature Importance Ranking:")
    for i, idx in enumerate(indices):
        print(f"{i+1}. {feature_names[idx]}: {importances[idx]:.4f}")

    print(f"\nPlot saved at: {plot_path}")

else:
    print("Feature importance not available")

## 8. Error Distribution by Parameter Ranges

In [ ]:
errors = np.abs(residuals)
rel_errors = (errors / y_test) * 100

print("Error Statistics by Parameter Ranges:")
print("="*70)

# ==============================
# Use original (unscaled) data
# ==============================
k_vals = X_data[:, 0]   # kL
h_vals = X_data[:, 1]   # L

# Match length with test set
k_vals = k_vals[:len(errors)]
h_vals = h_vals[:len(errors)]

# ------------------------------
# By kL
# ------------------------------
print(f"\nWave Number (kL) range: [{k_vals.min():.3f}, {k_vals.max():.3f}]")

k_mask_low = k_vals < np.percentile(k_vals, 50)

print(f"  Low kL:  MAE = {errors[k_mask_low].mean():.6e}, MAPE = {rel_errors[k_mask_low].mean():.2f}%")
print(f"  High kL: MAE = {errors[~k_mask_low].mean():.6e}, MAPE = {rel_errors[~k_mask_low].mean():.2f}%")

# ------------------------------
# By L
# ------------------------------
print(f"\nLayer Thickness (L) range: [{h_vals.min():.3f}, {h_vals.max():.3f}]")

h_mask_low = h_vals < np.percentile(h_vals, 50)

print(f"  Low L:  MAE = {errors[h_mask_low].mean():.6e}, MAPE = {rel_errors[h_mask_low].mean():.2f}%")
print(f"  High L: MAE = {errors[~h_mask_low].mean():.6e}, MAPE = {rel_errors[~h_mask_low].mean():.2f}%")

print("="*70)

## 9. Summary & Conclusion

In [ ]:
print("\n" + "="*70)
print("MODEL EVALUATION SUMMARY")
print("="*70)

print("\nAccuracy Metrics:")
print(f"R² Score : {r2:.4f} ({r2*100:.2f}% variance explained)")
print(f"MAE      : {mae:.6e}")
print(f"RMSE     : {rmse:.6e}")
print(f"MAPE     : {mape:.4f}%")

print("\nModel Performance Assessment:")
if r2 > 0.95:
    print("Excellent agreement between ML predictions and analytical results.")
elif r2 > 0.85:
    print("Very good predictive performance with minor deviations.")
elif r2 > 0.7:
    print("Reasonable performance; some discrepancies observed.")
else:
    print("Model accuracy is limited; further improvement is required.")

print("\nObservations:")
print("• The model successfully captures the nonlinear dispersion behavior.")
print("• Errors remain low across the considered parameter range.")
print("• The surrogate model provides a computationally efficient alternative to analytical solutions.")

print("\n" + "="*70)